# Exploratory Data Analysis of PubMed RCT 20k Dataset

This notebook performs exploratory data analysis on the PubMed RCT 20k dataset for biomedical NLP sentence classification.

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from datasets import load_dataset

from data_loader import load_pubmed_rct_data, get_class_labels
from preprocess import preprocess_sentences, get_sentence_lengths

# Set style
plt.style.use('default')
sns.set_palette('husl')

## Load Dataset

Load the PubMed RCT 20k dataset and examine basic statistics.

In [ ]:
# Load dataset
print("Loading PubMed RCT 20k dataset...")
dataset = load_pubmed_rct_data()
class_labels = get_class_labels()

print(f"Dataset splits: {list(dataset.keys())}")
print(f"Class labels: {class_labels}")

# Basic statistics
for split in ['train', 'validation', 'test']:
    print(f"\n{split.capitalize()} set:")
    print(f"  Number of samples: {len(dataset[split])}")
    labels = dataset[split]['label']
    unique, counts = np.unique(labels, return_counts=True)
    for label, count in zip(unique, counts):
        print(f"  {class_labels[label]}: {count} ({count/len(labels)*100:.1f}%)")

## Class Distribution

Visualize the distribution of classes across different splits.

In [ ]:
# Class distribution visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, split in enumerate(['train', 'validation', 'test']):
    labels = dataset[split]['label']
    unique, counts = np.unique(labels, return_counts=True)
    class_names = [class_labels[label] for label in unique]
    
    axes[i].bar(class_names, counts)
    axes[i].set_title(f'{split.capitalize()} Set Class Distribution')
    axes[i].set_ylabel('Number of Samples')
    axes[i].tick_params(axis='x', rotation=45)
    
    # Add percentages on bars
    for j, count in enumerate(counts):
        percentage = count / len(labels) * 100
        axes[i].text(j, count + max(counts)*0.01, f'{percentage:.1f}%', 
                    ha='center', va='bottom')

plt.tight_layout()
plt.savefig('../figures/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## Sentence Length Analysis

Analyze the distribution of sentence lengths (number of words) across classes.

In [ ]:
# Sentence length analysis
train_texts = preprocess_sentences(dataset['train']['text'])
train_labels = dataset['train']['label']
train_lengths = get_sentence_lengths(train_texts)

# Create DataFrame for easier analysis
df_lengths = pd.DataFrame({
    'length': train_lengths,
    'class': [class_labels[label] for label in train_labels]
})

print("Sentence length statistics:")
print(df_lengths.groupby('class')['length'].describe())

# Visualize sentence lengths by class
plt.figure(figsize=(12, 8))
sns.boxplot(data=df_lengths, x='class', y='length')
plt.title('Sentence Length Distribution by Class')
plt.ylabel('Number of Words')
plt.xlabel('Class')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../figures/sentence_lengths_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

# Histogram of sentence lengths
plt.figure(figsize=(10, 6))
plt.hist(train_lengths, bins=50, alpha=0.7, edgecolor='black')
plt.title('Distribution of Sentence Lengths')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.axvline(np.mean(train_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(train_lengths):.1f}')
plt.axvline(np.median(train_lengths), color='green', linestyle='--', label=f'Median: {np.median(train_lengths):.1f}')
plt.legend()
plt.tight_layout()
plt.savefig('../figures/sentence_lengths_histogram.png', dpi=300, bbox_inches='tight')
plt.show()

## Example Sentences

Show example sentences from each class to understand the data better.

In [ ]:
# Show example sentences from each class
print("Example sentences from each class:\n")

for class_idx, class_name in class_labels.items():
    # Find examples from this class
    indices = [i for i, label in enumerate(dataset['train']['label']) if label == class_idx]
    
    print(f"### {class_name}")
    # Show 3 examples
    for i in range(min(3, len(indices))):
        idx = indices[i]
        text = dataset['train']['text'][idx]
        print(f"{i+1}. {text}")
    print()

## Text Statistics

Basic text statistics and vocabulary analysis.

In [ ]:
# Basic text statistics
all_texts = preprocess_sentences(dataset['train']['text'])
all_words = [word for text in all_texts for word in text.split()]

print("Text Statistics:")
print(f"Total sentences: {len(all_texts)}")
print(f"Total words: {len(all_words)}")
print(f"Unique words: {len(set(all_words))}")
print(f"Average words per sentence: {len(all_words)/len(all_texts):.2f}")
print(f"Max words in a sentence: {max(len(text.split()) for text in all_texts)}")
print(f"Min words in a sentence: {min(len(text.split()) for text in all_texts)}")

# Most common words
from collections import Counter
word_freq = Counter(all_words)
print(f"\nMost common words:")
for word, freq in word_freq.most_common(20):
    print(f"  {word}: {freq}")

## Summary

This EDA provides insights into:
- Balanced class distribution across splits
- Sentence length characteristics (most sentences are short)
- Examples of different rhetorical roles in clinical abstracts
- Basic vocabulary statistics

The dataset appears well-suited for sentence classification tasks with clear distinctions between the rhetorical roles.